# Song Release Year Prediction — End-to-End Regression Pipeline

**Tugas UTS — Machine Learning & Deep Learning**

---

## Deskripsi
Notebook ini membangun pipeline regresi end-to-end untuk memprediksi **tahun rilis lagu** berdasarkan fitur audio.

### Alur Pipeline:
1. Data Loading & EDA
2. Preprocessing (Missing Values, Outliers, Scaling)
3. Feature Engineering & Selection
4. Model Training (Linear Regression, Ridge, Random Forest, XGBoost)
5. Hyperparameter Tuning dengan **Optuna**
6. Evaluasi (MSE, RMSE, MAE, R²)
7. Interpretasi dengan **LIME**
8. Experiment Tracking dengan **MLFlow**

## 1. Import Libraries

In [ ]:
# ── Standard Libraries ──
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Sklearn ──
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.pipeline import Pipeline

# ── Optuna ──
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── MLFlow ──
import mlflow
import mlflow.sklearn

# ── LIME ──
import lime
import lime.lime_tabular

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print('Semua library berhasil di-import!')

## 2. Data Loading & Exploratory Data Analysis (EDA)

In [ ]:
# ── Load Dataset ──
# Dataset tidak memiliki header, kolom pertama adalah target (tahun rilis)
df = pd.read_csv('midterm-regresi-dataset.csv', header=None)

# Beri nama kolom
n_features = df.shape[1] - 1
col_names = ['year'] + [f'feature_{i}' for i in range(1, n_features + 1)]
df.columns = col_names

print(f'Shape Dataset: {df.shape}')
print(f'Target (year): {df["year"].min()} - {df["year"].max()}')
print(f'Jumlah Fitur: {n_features}')
df.head()

In [ ]:
# ── Informasi Dataset ──
print('=== INFO DATASET ===')
df.info()
print()
print('=== STATISTIK DESKRIPTIF ===')
df.describe()

In [ ]:
# ── Cek Missing Values ──
missing = df.isnull().sum().sum()
print(f'Total Missing Values: {missing}')
print('Dataset bersih, tidak ada missing values!' if missing == 0 else '⚠️ Ada missing values!')

In [ ]:
# ── Distribusi Target (Tahun Rilis) ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['year'], bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribusi Tahun Rilis Lagu', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Tahun')
axes[0].set_ylabel('Jumlah Lagu')

# Value counts per decade
decade = (df['year'] // 10 * 10).value_counts().sort_index()
axes[1].bar(decade.index.astype(str), decade.values, color='coral', edgecolor='white')
axes[1].set_title('Jumlah Lagu per Dekade', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Dekade')
axes[1].set_ylabel('Jumlah Lagu')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('eda_target_distribution.png', bbox_inches='tight')
plt.show()
print('\nInsight: Dataset didominasi lagu dari era 2000-an')

In [ ]:
# ── Korelasi Fitur dengan Target ──
# Ambil 20 fitur dengan korelasi tertinggi terhadap year
correlations = df.corr()['year'].drop('year').abs().sort_values(ascending=False)
top20 = correlations.head(20)

plt.figure(figsize=(10, 7))
top20.plot(kind='barh', color='teal', edgecolor='white')
plt.title('Top 20 Fitur dengan Korelasi Tertinggi terhadap Tahun Rilis', fontsize=13, fontweight='bold')
plt.xlabel('Korelasi Absolut')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('eda_feature_correlation.png', bbox_inches='tight')
plt.show()

print('Top 5 Fitur Paling Berkorelasi:')
print(top20.head())

In [ ]:
# ── Boxplot Outlier Detection (sample fitur) ──
sample_features = [f'feature_{i}' for i in range(1, 11)]
fig, ax = plt.subplots(figsize=(14, 5))
df[sample_features].boxplot(ax=ax, vert=False)
ax.set_title('Boxplot 10 Fitur Pertama (Deteksi Outlier)', fontsize=13, fontweight='bold')
ax.set_xlabel('Nilai')
plt.tight_layout()
plt.savefig('eda_boxplot.png', bbox_inches='tight')
plt.show()

## 3. Preprocessing

In [ ]:
# ── Pisahkan Fitur dan Target ──
X = df.drop(columns=['year'])
y = df['year']

print(f'Fitur (X): {X.shape}')
print(f'Target (y): {y.shape}')

In [ ]:
# ── Capping Outlier dengan IQR (clip per fitur) ──
# Gunakan sampling 100k baris agar proses lebih cepat
print(' Melakukan capping outlier dengan metode IQR...')

Q1 = X.quantile(0.01)
Q3 = X.quantile(0.99)
X_clipped = X.clip(lower=Q1, upper=Q3, axis=1)

print('Outlier capping selesai!')

In [ ]:
# ── Train-Test Split (80:20) ──
X_train, X_test, y_train, y_test = train_test_split(
    X_clipped, y, test_size=0.2, random_state=42
)

print(f'Train size : {X_train.shape}')
print(f'Test size  : {X_test.shape}')

In [ ]:
# ── Feature Scaling dengan StandardScaler ──
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Scaling selesai! Mean ≈ 0, Std ≈ 1 pada data train')

## 4. Feature Engineering & Selection

In [ ]:
# ── Pilih Top-K Fitur dengan SelectKBest ──
K = 50  # Gunakan 50 fitur terbaik
selector = SelectKBest(score_func=f_regression, k=K)
X_train_sel = selector.fit_transform(X_train_scaled, y_train)
X_test_sel  = selector.transform(X_test_scaled)

# Nama fitur yang dipilih
selected_mask = selector.get_support()
selected_features = X.columns[selected_mask].tolist()

print(f'Fitur terpilih: {len(selected_features)} dari {X.shape[1]}')
print(f'Top 10 fitur: {selected_features[:10]}')

## 5. Model Training (Baseline)

In [ ]:
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    """Fit model dan kembalikan dict metrik evaluasi."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    mse  = mean_squared_error(y_te, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_te, y_pred)
    r2   = r2_score(y_te, y_pred)
    print(f'[{name}] RMSE={rmse:.4f} | MAE={mae:.4f} | R²={r2:.4f}')
    return {'Model': name, 'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2}, model, y_pred

results = []

print('=== BASELINE MODELS ===')

# 1. Linear Regression
r, lr_model, lr_pred = evaluate_model(
    'Linear Regression', LinearRegression(),
    X_train_sel, y_train, X_test_sel, y_test)
results.append(r)

# 2. Ridge Regression
r, ridge_model, ridge_pred = evaluate_model(
    'Ridge Regression', Ridge(alpha=1.0),
    X_train_sel, y_train, X_test_sel, y_test)
results.append(r)

# 3. Random Forest (n_estimators kecil untuk kecepatan)
r, rf_model, rf_pred = evaluate_model(
    'Random Forest', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    X_train_sel, y_train, X_test_sel, y_test)
results.append(r)

# 4. Gradient Boosting
r, gb_model, gb_pred = evaluate_model(
    'Gradient Boosting', GradientBoostingRegressor(n_estimators=100, random_state=42),
    X_train_sel, y_train, X_test_sel, y_test)
results.append(r)

In [ ]:
# ── Tampilkan Tabel Perbandingan ──
results_df = pd.DataFrame(results).sort_values('RMSE')
print('\n=== PERBANDINGAN MODEL (BASELINE) ===')
results_df.style.highlight_min(subset=['RMSE','MAE','MSE'], color='lightgreen') \
                .highlight_max(subset=['R2'], color='lightgreen') \
                .format({'MSE':'{:.2f}','RMSE':'{:.4f}','MAE':'{:.4f}','R2':'{:.4f}'})

In [ ]:
# ── Visualisasi Perbandingan Metrik ──
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = ['RMSE', 'MAE', 'R2']
colors  = ['#e74c3c', '#3498db', '#2ecc71']

for ax, metric, color in zip(axes, metrics, colors):
    ax.bar(results_df['Model'], results_df[metric], color=color, edgecolor='white')
    ax.set_title(f'Perbandingan {metric}', fontweight='bold')
    ax.set_xticklabels(results_df['Model'], rotation=15, ha='right')
    ax.set_ylabel(metric)

plt.suptitle('Perbandingan Model Baseline', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('baseline_model_comparison.png', bbox_inches='tight')
plt.show()

## 6. Hyperparameter Tuning dengan Optuna

In [ ]:
# ── Optuna: Tuning Random Forest ──
print('Memulai Optuna hyperparameter tuning untuk Random Forest...')

def objective_rf(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 50, 300),
        'max_depth'        : trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features'     : trial.suggest_float('max_features', 0.3, 1.0),
        'random_state'     : 42,
        'n_jobs'           : -1,
    }
    model = RandomForestRegressor(**params)
    model.fit(X_train_sel, y_train)
    y_pred = model.predict(X_test_sel)
    return np.sqrt(mean_squared_error(y_test, y_pred))  # minimize RMSE

study_rf = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=42),
    study_name='RandomForest_Tuning'
)
study_rf.optimize(objective_rf, n_trials=20, show_progress_bar=True)

print(f'\nOptuna selesai!')
print(f'Best RMSE : {study_rf.best_value:.4f}')
print(f'Best Params: {study_rf.best_params}')

In [ ]:
# ── Visualisasi Optuna ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Optimization history
trial_nums  = [t.number for t in study_rf.trials]
trial_vals  = [t.value  for t in study_rf.trials]
best_so_far = [min(trial_vals[:i+1]) for i in range(len(trial_vals))]

axes[0].plot(trial_nums, trial_vals,    'o-', alpha=0.5, label='RMSE per trial')
axes[0].plot(trial_nums, best_so_far,   's-', color='red', label='Best so far')
axes[0].set_title('Optuna: Optimization History', fontweight='bold')
axes[0].set_xlabel('Trial Number')
axes[0].set_ylabel('RMSE')
axes[0].legend()

# Param importance (manual)
param_names = list(study_rf.best_params.keys())
importances = [abs(study_rf.best_params[p]) if isinstance(study_rf.best_params[p], float)
               else study_rf.best_params[p] for p in param_names]
axes[1].barh(param_names, importances, color='teal')
axes[1].set_title('Best Hyperparameter Values', fontweight='bold')
axes[1].set_xlabel('Value')

plt.suptitle('Optuna Tuning — Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('optuna_result.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Train Model Terbaik (Best Params dari Optuna) ──
best_rf = RandomForestRegressor(**study_rf.best_params)
best_rf.fit(X_train_sel, y_train)
y_pred_best = best_rf.predict(X_test_sel)

mse_best  = mean_squared_error(y_test, y_pred_best)
rmse_best = np.sqrt(mse_best)
mae_best  = mean_absolute_error(y_test, y_pred_best)
r2_best   = r2_score(y_test, y_pred_best)

print('=== HASIL MODEL TERBAIK (Random Forest + Optuna) ===')
print(f'MSE  : {mse_best:.4f}')
print(f'RMSE : {rmse_best:.4f}')
print(f'MAE  : {mae_best:.4f}')
print(f'R²   : {r2_best:.4f}')

## 7. Evaluasi Model

In [ ]:
# ── Actual vs Predicted Plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
sample_idx = np.random.choice(len(y_test), 2000, replace=False)
axes[0].scatter(y_test.values[sample_idx], y_pred_best[sample_idx],
                alpha=0.3, color='steelblue', s=10)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             'r--', lw=2, label='Ideal')
axes[0].set_title('Actual vs Predicted Year', fontweight='bold')
axes[0].set_xlabel('Actual Year')
axes[0].set_ylabel('Predicted Year')
axes[0].legend()

# Residual plot
residuals = y_test.values - y_pred_best
axes[1].scatter(y_pred_best[sample_idx], residuals[sample_idx],
                alpha=0.3, color='coral', s=10)
axes[1].axhline(0, color='red', lw=2, linestyle='--')
axes[1].set_title('Residual Plot', fontweight='bold')
axes[1].set_xlabel('Predicted Year')
axes[1].set_ylabel('Residual (Actual - Predicted)')

plt.suptitle(f'Evaluasi Model — RMSE: {rmse_best:.4f} | R²: {r2_best:.4f}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('model_evaluation.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature Importance (dari Random Forest terbaik) ──
feat_importances = pd.Series(
    best_rf.feature_importances_,
    index=selected_features
).sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 6))
feat_importances.plot(kind='barh', color='darkorange', edgecolor='white')
plt.title('Top 20 Feature Importances (Random Forest)', fontweight='bold')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Tabel Perbandingan Semua Model ──
final_results = results_df.copy()
final_results = pd.concat([final_results, pd.DataFrame([{
    'Model': 'RF + Optuna (Best)',
    'MSE': mse_best,
    'RMSE': rmse_best,
    'MAE': mae_best,
    'R2': r2_best
}])], ignore_index=True).sort_values('RMSE')

print('=== PERBANDINGAN SEMUA MODEL ===')
final_results.style \
    .highlight_min(subset=['RMSE','MAE','MSE'], color='lightgreen') \
    .highlight_max(subset=['R2'], color='lightgreen') \
    .format({'MSE':'{:.2f}','RMSE':'{:.4f}','MAE':'{:.4f}','R2':'{:.4f}'})

## 8. Interpretasi dengan LIME

In [ ]:
# ── Setup LIME Explainer ──
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_sel,
    feature_names=selected_features,
    mode='regression',
    random_state=42
)

print('LIME explainer siap!')

In [ ]:
# ── Jelaskan Prediksi untuk 3 Sample ──
sample_indices = [0, 100, 500]   # index di X_test_sel

for i, idx in enumerate(sample_indices):
    exp = explainer.explain_instance(
        data_row     = X_test_sel[idx],
        predict_fn   = best_rf.predict,
        num_features = 10
    )

    actual    = y_test.values[idx]
    predicted = best_rf.predict(X_test_sel[idx:idx+1])[0]

    print(f'\n─── Sample #{i+1} ───')
    print(f'Actual Year   : {actual}')
    print(f'Predicted Year: {predicted:.1f}')
    print(f'Error         : {abs(actual - predicted):.2f} tahun')

    fig = exp.as_pyplot_figure()
    fig.suptitle(f'LIME — Sample #{i+1} | Actual: {actual} | Pred: {predicted:.1f}',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'lime_explanation_sample{i+1}.png', bbox_inches='tight')
    plt.show()

### Interpretasi LIME
- LIME membuat model lokal (linear) di sekitar satu prediksi untuk menjelaskan *mengapa* model menghasilkan prediksi tersebut.
- Fitur berwarna **hijau** = mendorong prediksi ke nilai yang lebih tinggi (tahun lebih baru)
- Fitur berwarna **merah** = mendorong prediksi ke nilai yang lebih rendah (tahun lebih lama)
- Semakin panjang bar, semakin besar kontribusi fitur tersebut terhadap prediksi.

## 9. Experiment Tracking dengan MLFlow

In [ ]:
# ── Setup MLFlow ──
mlflow.set_tracking_uri('file:./mlruns')  # Simpan lokal
mlflow.set_experiment('Song-Year-Regression')

print('MLFlow experiment siap!')

In [ ]:
# ── Log Semua Model ke MLFlow ──
model_registry = [
    ('Linear Regression', lr_model,    lr_pred,    {}),
    ('Ridge Regression',  ridge_model, ridge_pred, {'alpha': 1.0}),
    ('Random Forest',     rf_model,    rf_pred,    {'n_estimators': 100}),
    ('Gradient Boosting', gb_model,    gb_pred,    {'n_estimators': 100}),
    ('RF + Optuna',       best_rf,     y_pred_best, study_rf.best_params),
]

for model_name, model_obj, pred, params in model_registry:
    with mlflow.start_run(run_name=model_name):
        # Log params
        mlflow.log_params(params)
        mlflow.log_param('n_features_selected', K)
        mlflow.log_param('train_size', X_train_sel.shape[0])
        mlflow.log_param('test_size',  X_test_sel.shape[0])

        # Log metrics
        mse  = mean_squared_error(y_test, pred)
        rmse = np.sqrt(mse)
        mae  = mean_absolute_error(y_test, pred)
        r2   = r2_score(y_test, pred)

        mlflow.log_metric('mse',  mse)
        mlflow.log_metric('rmse', rmse)
        mlflow.log_metric('mae',  mae)
        mlflow.log_metric('r2',   r2)

        # Log model
        mlflow.sklearn.log_model(model_obj, artifact_path='model')

        # Log artifacts (gambar)
        for img in ['eda_target_distribution.png',
                    'eda_feature_correlation.png',
                    'baseline_model_comparison.png',
                    'optuna_result.png',
                    'model_evaluation.png',
                    'feature_importance.png']:
            try:
                mlflow.log_artifact(img)
            except Exception:
                pass

        print(f'[MLFlow] Logged: {model_name} | RMSE={rmse:.4f} | R²={r2:.4f}')

print('\nSemua run berhasil di-log ke MLFlow!')
print('Jalankan `mlflow ui` di terminal untuk melihat dashboard.')

## 10. Kesimpulan & Ringkasan

In [ ]:
# ── Ringkasan Akhir ──
print('=' * 55)
print('       RINGKASAN AKHIR — SONG YEAR PREDICTION')
print('=' * 55)
print(f'Dataset   : {df.shape[0]:,} lagu | {n_features} fitur audio')
print(f'Target    : Tahun rilis ({df["year"].min()}–{df["year"].max()})')
print(f'Fitur sel : {K} (dari {n_features}) via SelectKBest')
print()
print('─── Model Terbaik: Random Forest + Optuna ───')
print(f'  RMSE : {rmse_best:.4f} tahun')
print(f'  MAE  : {mae_best:.4f} tahun')
print(f'  R²   : {r2_best:.4f}')
print()
print('─── Insight ───')
print(f'  • Rata-rata error prediksi ≈ {mae_best:.1f} tahun')
print(f'  • Model mampu menjelaskan {r2_best*100:.1f}% variansi tahun rilis')
print(f'  • Optuna meningkatkan performa dari baseline Random Forest')
print()
print('  → Coba XGBoost/LightGBM untuk hasil yang lebih baik!')
print('=' * 55)